# 🌳 Étape 3 — Ablation Study + Figures finales + Tests statistiques
## Production des résultats clés pour le manuscrit

**À exécuter APRÈS** le notebook unifié Étape 1+2 (les variables `df`, `FEATURES_ALL`, `FEATURES_BASE`, `FEATURES_TEMPORAL`, `make_models()` doivent être en mémoire).

**Contenu :**
1. Ablation Study complète : Statique vs Temporel vs All, sur toutes les semaines
2. Figure 8 corrigée : régions stable/unstable
3. Figure 10 : Ablation curve (LE résultat phare du papier)
4. Tableau 3 : Comparaison statistique (Wilcoxon)
5. Reduced Feature Set : top-5 SHAP performance
6. Export tableaux LaTeX-ready pour le manuscrit

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — Ablation Study : Static vs Temporal vs All
# ═══════════════════════════════════════════════════════════════
#
# Pour CHAQUE semaine, on entraîne 3 variantes du même modèle XGBoost :
#   - STATIC ONLY : 8 features statiques
#   - TEMPORAL ONLY : 16 features temporelles
#   - ALL : 24 features (statique + temporel)
#
# Question : à quel moment les features temporelles dépassent-elles
# les statiques, et de combien ?
# ═══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import f1_score, make_scorer

scorer = make_scorer(f1_score, average='macro')
weeks_to_test = list(range(3, int(df['week_num'].max()) + 1))

ablation_results = []

print(f'Ablation Study — {len(weeks_to_test)} semaines × 3 configurations × XGBoost')
print('(cela peut prendre 5-15 min)')
print()

for w in weeks_to_test:
    mask = df['week_num'] <= w
    if mask.sum() < 30:
        continue
    y_w = df.loc[mask, 'label_3'].values
    g_w = df.loc[mask, 'Olive_ID'].values

    if len(np.unique(y_w)) < 3 or len(np.unique(g_w)) < 10:
        continue

    row = {'week': w, 'n_obs': int(mask.sum())}

    for variant_name, feats in [
        ('static',   FEATURES_BASE),
        ('temporal', FEATURES_TEMPORAL),
        ('all',      FEATURES_ALL)
    ]:
        X_w = df.loc[mask, feats].values
        pipe = make_models()['XGBoost']
        cv = cross_validate(
            pipe, X_w, y_w,
            cv=GroupKFold(n_splits=5).split(X_w, y_w, g_w),
            scoring=scorer, n_jobs=1
        )
        row[f'{variant_name}_f1']     = cv['test_score'].mean()
        row[f'{variant_name}_f1_std'] = cv['test_score'].std()
        row[f'{variant_name}_folds']  = cv['test_score'].tolist()

    ablation_results.append(row)
    print(f'  Semaine ≤ {w:2d} | '
          f'Static={row["static_f1"]:.3f}±{row["static_f1_std"]:.3f} | '
          f'Temporal={row["temporal_f1"]:.3f}±{row["temporal_f1_std"]:.3f} | '
          f'All={row["all_f1"]:.3f}±{row["all_f1_std"]:.3f}')

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv('Ablation_Study_Results.csv', index=False)
print('\n✅ Cellule 1 — Ablation Study terminée')
print('   Fichier : Ablation_Study_Results.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 — Figure 10 : ABLATION CURVE (résultat phare du papier)
# ═══════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(11, 6.5))

weeks_x = df_ablation['week'].values

# Trois courbes
ax.plot(weeks_x, df_ablation['static_f1'],
        marker='o', linewidth=2.5, markersize=8,
        color='#7f8c8d', label='Static features only (8)')
ax.fill_between(weeks_x,
                df_ablation['static_f1'] - df_ablation['static_f1_std'],
                df_ablation['static_f1'] + df_ablation['static_f1_std'],
                alpha=0.15, color='#7f8c8d')

ax.plot(weeks_x, df_ablation['temporal_f1'],
        marker='s', linewidth=2.5, markersize=8,
        color='#e74c3c', label='Temporal features only (16)')
ax.fill_between(weeks_x,
                df_ablation['temporal_f1'] - df_ablation['temporal_f1_std'],
                df_ablation['temporal_f1'] + df_ablation['temporal_f1_std'],
                alpha=0.15, color='#e74c3c')

ax.plot(weeks_x, df_ablation['all_f1'],
        marker='^', linewidth=3.0, markersize=9,
        color='#27ae60', label='All features (24)')
ax.fill_between(weeks_x,
                df_ablation['all_f1'] - df_ablation['all_f1_std'],
                df_ablation['all_f1'] + df_ablation['all_f1_std'],
                alpha=0.20, color='#27ae60')

# Seuils
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(0.80, color='gray', linestyle=':',  alpha=0.5, linewidth=1)
ax.text(weeks_x[-1] + 0.3, 0.70, 'F1=0.70', fontsize=9, color='gray', va='center')
ax.text(weeks_x[-1] + 0.3, 0.80, 'F1=0.80', fontsize=9, color='gray', va='center')

# Région stable / unstable
stable_threshold = 0.05
stable_weeks = df_ablation[df_ablation['all_f1_std'] <= stable_threshold]['week'].values
if len(stable_weeks) > 0:
    w_stable = int(stable_weeks.min())
    ax.axvspan(weeks_x.min() - 0.5, w_stable - 0.5,
               alpha=0.07, color='red', label=f'Unstable region (std > 0.05)')
    ax.axvspan(w_stable - 0.5, weeks_x.max() + 0.5,
               alpha=0.07, color='green', label=f'Stable region (std ≤ 0.05)')

ax.set_xlabel('Cumulative weeks of measurements', fontsize=12)
ax.set_ylabel('Macro F1-score (5-fold GroupKFold CV, 3 classes)', fontsize=12)
ax.set_title('Figure 10 — Ablation Study: Impact of Temporal Feature Engineering',
             fontsize=13, fontweight='bold')
ax.set_ylim(0.2, 1.0)
ax.set_xticks(weeks_x)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=10, framealpha=0.95)

plt.tight_layout()
plt.savefig('Figure10_Ablation_Curve.png', dpi=300, bbox_inches='tight')
plt.savefig('Figure10_Ablation_Curve.pdf', bbox_inches='tight')  # for LaTeX
plt.show()
print('✅ Figure 10 sauvegardée (PNG + PDF)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — Tests statistiques (Wilcoxon)
# ═══════════════════════════════════════════════════════════════
#
# H0 : pas de différence entre les deux variantes
# H1 : une variante est significativement meilleure
#
# Wilcoxon signed-rank sur les F1 par semaine (paired).
# ═══════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

print('=' * 70)
print('  TESTS STATISTIQUES — Wilcoxon signed-rank (paired by week)')
print('=' * 70)
print()

tests_to_run = [
    ('Temporal vs Static', 'temporal_f1', 'static_f1'),
    ('All vs Static',      'all_f1',      'static_f1'),
    ('All vs Temporal',    'all_f1',      'temporal_f1')
]

stats_results = []
for name, col_a, col_b in tests_to_run:
    a = df_ablation[col_a].values
    b = df_ablation[col_b].values
    try:
        stat, p = wilcoxon(a, b, alternative='greater')
        mean_diff = (a - b).mean()
        median_diff = np.median(a - b)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'  {name:25s} | Δmean = {mean_diff:+.3f} | Δmedian = {median_diff:+.3f} | p = {p:.4f} {sig}')
        stats_results.append({
            'Comparison'    : name,
            'Delta_mean_F1' : round(mean_diff, 3),
            'Delta_median_F1': round(median_diff, 3),
            'Wilcoxon_stat' : round(stat, 2),
            'p_value'       : round(p, 6),
            'Significance'  : sig
        })
    except ValueError as e:
        print(f'  {name:25s} | Erreur : {e}')

df_stats = pd.DataFrame(stats_results)
df_stats.to_csv('Tableau_Tests_Statistiques.csv', index=False)

print()
print('Légende : *** p<0.001 | ** p<0.01 | * p<0.05 | ns = non significatif')
print()
print('✅ Cellule 3 — Tests statistiques terminés')
print('   Fichier : Tableau_Tests_Statistiques.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 4 — Reduced Feature Set (top-5 SHAP)
# ═══════════════════════════════════════════════════════════════
#
# Contribution "low-cost precision agriculture" :
# Est-ce qu'on peut atteindre une bonne performance avec seulement
# 5 features (= 5 capteurs / mesures à faire sur le terrain) ?
# ═══════════════════════════════════════════════════════════════

# Récupérer le top 5 SHAP de l'étape 1 (shap_ranked existe déjà)
# Sinon, fallback sur l'importance XGBoost
try:
    top5_features = shap_ranked.head(5)['Feature'].tolist()
    print(f'Top 5 features (SHAP) : {top5_features}\n')
except NameError:
    # Fallback : importance XGBoost
    pipe = make_models()['XGBoost']
    pipe.fit(X_all, y_3)
    clf = pipe.named_steps['clf']
    imp_df = pd.DataFrame({
        'Feature': FEATURES_ALL,
        'Importance': clf.feature_importances_
    }).sort_values('Importance', ascending=False)
    top5_features = imp_df.head(5)['Feature'].tolist()
    print(f'Top 5 features (XGBoost importance) : {top5_features}\n')

# Performance avec top-5 features
X_top5 = df[top5_features].values
scorer = make_scorer(f1_score, average='macro')

results_reduced = []
for k in [3, 5, 10, len(FEATURES_ALL)]:
    if k > len(FEATURES_ALL):
        continue
    if k == len(FEATURES_ALL):
        feats_k = FEATURES_ALL
        label   = f'All features ({k})'
    else:
        try:
            feats_k = shap_ranked.head(k)['Feature'].tolist()
        except NameError:
            feats_k = imp_df.head(k)['Feature'].tolist()
        label = f'Top-{k} SHAP'

    X_k  = df[feats_k].values
    pipe = make_models()['XGBoost']
    cv = cross_validate(
        pipe, X_k, y_3,
        cv=GroupKFold(n_splits=5).split(X_k, y_3, groups),
        scoring=scorer, n_jobs=1
    )
    f1m, f1s = cv['test_score'].mean(), cv['test_score'].std()
    results_reduced.append({
        'Variant'  : label,
        'N_features': k,
        'Features' : ', '.join(feats_k),
        'Macro_F1' : round(f1m, 3),
        'Std'      : round(f1s, 3)
    })
    print(f'  {label:18s} | {k:2d} features | F1 = {f1m:.3f} ± {f1s:.3f}')

df_reduced = pd.DataFrame(results_reduced)
df_reduced.to_csv('Tableau_Reduced_Feature_Set.csv', index=False)
print('\n✅ Cellule 4 — Reduced Feature Set terminé')
print('   Fichier : Tableau_Reduced_Feature_Set.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5 — Figure 8 corrigée (stable vs unstable region)
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Figure 8 (revised) — Early Detection Curve with stability regions',
             fontsize=14, fontweight='bold')

# 4 classes
ax = axes[0]
wx, f1y, f1s = df_early_4['week'].values, df_early_4['macro_f1'].values, df_early_4['macro_f1_std'].values
ax.plot(wx, f1y, marker='o', linewidth=2.5, markersize=8, color='#2980b9', label='Macro F1')
ax.fill_between(wx, f1y - f1s, f1y + f1s, alpha=0.25, color='#2980b9', label='± 1 std')
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.6)
ax.axhline(0.80, color='gray', linestyle=':',  alpha=0.6)
stable_4 = df_early_4[df_early_4['macro_f1_std'] <= 0.05]['week'].values
if len(stable_4) > 0:
    w_st = int(stable_4.min())
    ax.axvspan(wx.min() - 0.5, w_st - 0.5, alpha=0.10, color='red')
    ax.axvspan(w_st - 0.5, wx.max() + 0.5, alpha=0.10, color='green')
    ax.text(w_st, 0.30, f'Stable\nfrom\nweek {w_st}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Cumulative weeks', fontsize=11)
ax.set_ylabel('Macro F1', fontsize=11)
ax.set_title('A — 4 classes (0%/25%/50%/100%)', fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_xticks(wx)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')

# 3 classes
ax = axes[1]
wx, f1y, f1s = df_early_3['week'].values, df_early_3['macro_f1'].values, df_early_3['macro_f1_std'].values
ax.plot(wx, f1y, marker='s', linewidth=2.5, markersize=8, color='#c0392b', label='Macro F1')
ax.fill_between(wx, f1y - f1s, f1y + f1s, alpha=0.25, color='#c0392b', label='± 1 std')
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.6)
ax.axhline(0.80, color='gray', linestyle=':',  alpha=0.6)
stable_3 = df_early_3[df_early_3['macro_f1_std'] <= 0.05]['week'].values
if len(stable_3) > 0:
    w_st = int(stable_3.min())
    ax.axvspan(wx.min() - 0.5, w_st - 0.5, alpha=0.10, color='red')
    ax.axvspan(w_st - 0.5, wx.max() + 0.5, alpha=0.10, color='green')
    ax.text(w_st, 0.30, f'Stable\nfrom\nweek {w_st}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Cumulative weeks', fontsize=11)
ax.set_ylabel('Macro F1', fontsize=11)
ax.set_title('B — 3 classes (Severe / Moderate / Control)', fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_xticks(wx)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('Figure8_Revised_Early_Detection.png', dpi=300, bbox_inches='tight')
plt.savefig('Figure8_Revised_Early_Detection.pdf', bbox_inches='tight')
plt.show()
print('✅ Figure 8 corrigée sauvegardée')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 6 — Export tableaux LaTeX-ready
# ═══════════════════════════════════════════════════════════════

def df_to_latex(df, caption, label, float_fmt='%.3f'):
    return df.to_latex(
        index=False,
        caption=caption,
        label=label,
        float_format=float_fmt,
        escape=False
    )

# Table 1 — Main benchmark
main_results = []
for cfg_name in ['A — Base (4 classes)', 'B — Enrichi (4 classes)', 'C — Enrichi (3 classes)']:
    cfg = all_results[cfg_name]
    best = max(cfg, key=lambda k: cfg[k]['Macro F1'].mean())
    main_results.append({
        'Configuration': cfg_name.replace('—','-'),
        'Best model'   : best,
        'Macro F1'     : f"{cfg[best]['Macro F1'].mean():.3f} ± {cfg[best]['Macro F1'].std():.3f}",
        'Accuracy'     : f"{cfg[best]['Accuracy'].mean():.3f} ± {cfg[best]['Accuracy'].std():.3f}",
        'Bal. Acc'     : f"{cfg[best]['Balanced Accuracy'].mean():.3f} ± {cfg[best]['Balanced Accuracy'].std():.3f}"
    })
df_main = pd.DataFrame(main_results)
df_main.to_csv('Table1_Main_Results.csv', index=False)

with open('Table1_Main_Results.tex', 'w') as f:
    f.write(df_to_latex(df_main,
        'Main benchmark across three feature configurations. Macro F1, Accuracy and Balanced Accuracy reported as mean ± std over 5-fold GroupKFold cross-validation (groups = Olive\\_ID).',
        'tab:main_results'))

# Table 2 — Ablation results (semaine finale)
df_abl_summary = df_ablation[['week', 'static_f1', 'static_f1_std',
                               'temporal_f1', 'temporal_f1_std',
                               'all_f1', 'all_f1_std']].tail(5).copy()
df_abl_summary.columns = ['Week', 'Static F1', 'Static std',
                          'Temporal F1', 'Temporal std',
                          'All F1', 'All std']
df_abl_summary.to_csv('Table2_Ablation_Summary.csv', index=False)
with open('Table2_Ablation_Summary.tex', 'w') as f:
    f.write(df_to_latex(df_abl_summary,
        'Ablation study: Macro F1 across feature configurations for the last 5 cumulative-week settings.',
        'tab:ablation'))

# Table 3 — Wilcoxon stats
df_stats.to_csv('Table3_Wilcoxon.csv', index=False)
with open('Table3_Wilcoxon.tex', 'w') as f:
    f.write(df_to_latex(df_stats,
        'Statistical comparison between feature configurations (Wilcoxon signed-rank, one-sided).',
        'tab:stats'))

# Table 4 — Reduced feature set
df_reduced.to_csv('Table4_Reduced_Features.csv', index=False)
with open('Table4_Reduced_Features.tex', 'w') as f:
    f.write(df_to_latex(df_reduced.drop(columns=['Features']),
        'Reduced feature set performance: Top-k features ranked by SHAP importance.',
        'tab:reduced'))

print('✅ 4 tableaux LaTeX et CSV exportés :')
for f in ['Table1_Main_Results', 'Table2_Ablation_Summary',
          'Table3_Wilcoxon', 'Table4_Reduced_Features']:
    print(f'   - {f}.csv et {f}.tex')

print()
print('=' * 65)
print('  RÉSUMÉ DES RÉSULTATS POUR LE MANUSCRIT')
print('=' * 65)
print()
f1_static_full   = df_ablation[df_ablation['week'] == df_ablation['week'].max()]['static_f1'].values[0]
f1_temporal_full = df_ablation[df_ablation['week'] == df_ablation['week'].max()]['temporal_f1'].values[0]
f1_all_full      = df_ablation[df_ablation['week'] == df_ablation['week'].max()]['all_f1'].values[0]
f1_static_w3     = df_ablation[df_ablation['week'] == 3]['static_f1'].values[0]
f1_temporal_w3   = df_ablation[df_ablation['week'] == 3]['temporal_f1'].values[0]

print(f'🔬 Résultat principal (3 classes, toutes semaines) :')
print(f'   Static  : F1 = {f1_static_full:.3f}')
print(f'   Temporal: F1 = {f1_temporal_full:.3f}  (+{f1_temporal_full-f1_static_full:+.3f})')
print(f'   All     : F1 = {f1_all_full:.3f}  (+{f1_all_full-f1_static_full:+.3f})')
print()
print(f'🔬 Résultat précoce (semaine ≤ 3) :')
print(f'   Static  : F1 = {f1_static_w3:.3f}')
print(f'   Temporal: F1 = {f1_temporal_w3:.3f}  (+{f1_temporal_w3-f1_static_w3:+.3f})')
print()
print(f'📊 Stable detection (std ≤ 0.05) :')
if len(stable_3) > 0:
    print(f'   3 classes : à partir de la semaine {int(stable_3.min())}')
if len(stable_4) > 0:
    print(f'   4 classes : à partir de la semaine {int(stable_4.min())}')
print()
print('✅ Étape 3 complète — Tous les fichiers prêts pour le manuscrit')